In [46]:
from pathlib import Path
import numpy as np
import pandas as pd


In [47]:
excel_path = Path('AFC_auto_coded.xlsx')
coded_sheet = 'Coded Data FINAL for regression'
coded = pd.read_excel(excel_path, sheet_name=coded_sheet)
len(coded), coded.head()


(285,
    Trade_Frequency  Day_Trade  Nstock_clean  Lottery_Stocks  Stress_Level  \
 0                1          0           0.0               0           5.0   
 1                5          0           3.0               0           5.5   
 2                2          1          10.0               1           5.5   
 3                5          0           3.0               1           5.5   
 4                3          0           8.0               1           2.0   
 
    Smoke  Drink_Alcohol  Addiction_Score  Gambling_Score  Male_Dummy  ...  \
 0      0              0                0            1.00           0  ...   
 1      0              1                2            1.25           1  ...   
 2      0              1                4            2.50           0  ...   
 3      0              0                0            1.00           1  ...   
 4      1              1                0            1.25           1  ...   
 
    Income  Master_Dummy  Risk_Tolerance_Score  Financ

In [48]:
binary = pd.DataFrame(index=coded.index)
summary_rows = []

def add_binary(name: str, series: pd.Series, note: str) -> None:
    series = series.astype('Int64')
    binary[name] = series
    valid = series.dropna()
    ones = int((valid == 1).sum())
    zeros = int((valid == 0).sum())
    total = int(valid.count())
    share = (ones / total) if total else np.nan
    summary_rows.append({
        'binary_variable_name': name,
        'ones': ones,
        'zeros': zeros,
        'total': total,
        'share_of_ones': share,
        'note': note,
    })

def flag_from_sets(series: pd.Series, ones=None, zeros=None) -> pd.Series:
    flag = pd.Series(pd.NA, index=series.index, dtype='Int64')
    if ones is not None:
        flag.loc[series.isin(ones)] = 1
    if zeros is not None:
        flag.loc[series.isin(zeros)] = 0
    return flag

def flag_from_condition(series: pd.Series, predicate) -> pd.Series:
    flag = pd.Series(pd.NA, index=series.index, dtype='Int64')
    mask = series.notna()
    flag.loc[mask] = predicate(series.loc[mask]).astype('Int64')
    return flag


In [49]:
# Trade frequency flags
add_binary('Recurrent_Trader', flag_from_sets(coded['Trade_Frequency'], {4, 5}, {1, 2, 3}), 'codes {4,5}=1, {1,2,3}=0')
#add_binary('Active_Trader (optional)', flag_from_sets(coded['Trade_Frequency'], {3, 4, 5}, {1, 2}), 'codes {3,4,5}=1, {1,2}=0')

# Direct binary copies
for col, name in [
    ('Day_Trade', 'Day_Trader'),
    ('Lottery_Stocks', 'Lottery_Stock_Trader'),
    ('Smoke', 'Smoker'),
    ('Drink_Alcohol', 'Alcohol_User'),
    ('Male_Dummy', 'Male_Investor'),
    ('Single_Dummy', 'Single_Investor'),
    ('Return_needed_for_exp', 'Needs_Returns_For_Expenses'),
    ('Derivative_Warrant', 'Derivative_User'),
    ('Margin_Account', 'Margin_User'),
    ('Master_Dummy', 'High_Education'),
]:
    add_binary(name, coded[col], f'Copied from {col}')

# Portfolio diversification
add_binary('Underdiversified_Portfolio', flag_from_condition(coded['Nstock_clean'], lambda s: s < 8), '<8 holdings → 1')

# Age-based splits
add_binary('Young_Investor', flag_from_condition(coded['Age'], lambda s: s <= 35), '<=35 → 1')
#add_binary('Senior_Investor (optional)', flag_from_condition(coded['Age'], lambda s: s >= 50), '>=50 → 1')

# Income brackets
add_binary('Low_Income', flag_from_condition(coded['Income'], lambda s: s <= 30000), '<=30k THB → 1')
#add_binary('High_Income (optional)', flag_from_condition(coded['Income'], lambda s: s >= 100000), '>=100k THB → 1')

# Overconfidence
add_binary('Overconfident_SelfView', flag_from_condition(coded['Percent_better_than_me'], lambda s: s <= 30), '<=30% think better -> 1')
#add_binary('Extreme_Overconfidence (optional)', flag_from_condition(coded['Percent_better_than_me'], lambda s: s <= 10), '<=10% think better -> 1')

# Experience
add_binary('Low_Experience', flag_from_sets(coded['Experience'], {1, 2}, {3, 4}), 'codes {1,2}=1, {3,4}=0')
#add_binary('High_Experience (optional)', flag_from_sets(coded['Experience'], {4}, {1, 2, 3}), 'code 4=1, {1,2,3}=0')

# Estimation interval quartile
est_p75 = coded['Est_Interval'].quantile(0.75)
add_binary('Highly_Overconfident_Interval', flag_from_condition(coded['Est_Interval'], lambda s, threshold=est_p75: s >= threshold), f'>= P75 ({est_p75:.2f})')

# Gambling tendencies
add_binary('Has_Gambling_Habit', flag_from_condition(coded['Gambling_Score'], lambda s: s >= 2.0), 'mean >=2.0 (1-5 scale)')
#add_binary('Extreme_Gambling_Tendency (optional)', flag_from_condition(coded['Gambling_Score'], lambda s: s >= 4), 'mean >=4 (1-5 scale)')

# Stress
add_binary('High_Stress', flag_from_condition(coded['Stress_Level'], lambda s: s >= 5), 'values 5-7 -> 1')
#add_binary('Low_Stress (optional)', flag_from_condition(coded['Stress_Level'], lambda s: s <= 2), 'values 1-2 -> 1')

# Risk tolerance
add_binary('High_Risk_Tolerance', flag_from_condition(coded['Risk_Tolerance_Score'], lambda s: s >= 7), '>=7 (scale 1-10)')
#add_binary('Very_High_Risk_Tolerance (optional)', flag_from_condition(coded['Risk_Tolerance_Score'], lambda s: s >= 8), '>=8 (scale 1-10)')

# Self-reported addiction: treat any positive score as yes
add_binary('Trading_Addiction_SelfReport', flag_from_condition(coded['Addiction_Score'], lambda s: s >= 1), 'Addiction_Score >=1 -> 1')

# Financial literacy
add_binary('Low_Financial_Literacy', flag_from_condition(coded['Financial_Literacy'], lambda s: s <= 6), '<=6 correct -> 1')
#add_binary('High_Financial_Literacy (optional)', flag_from_condition(coded['Financial_Literacy'], lambda s: s >= 8), '>=8 correct -> 1')

summary = pd.DataFrame(summary_rows)
binary.head()


,Recurrent_Trader,Day_Trader,Lottery_Stock_Trader,Smoker,Alcohol_User,Male_Investor,Single_Investor,Needs_Returns_For_Expenses,Derivative_User,Margin_User,...,Young_Investor,Low_Income,Overconfident_SelfView,Low_Experience,Highly_Overconfident_Interval,Has_Gambling_Habit,High_Stress,High_Risk_Tolerance,Trading_Addiction_SelfReport,Low_Financial_Literacy
0,0,0,0,0,0,0,1,0,0,0,...,1,0,0,1,1,0,1,0,0,1
1,1,0,0,0,1,1,1,1,0,0,...,1,1,0,1,0,0,1,0,1,1
2,0,1,1,0,1,0,1,1,1,0,...,1,1,1,0,1,1,1,1,1,0
3,1,0,1,0,0,1,0,0,1,0,...,1,1,0,0,0,0,1,0,0,0
4,0,0,1,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [50]:
summary


,binary_variable_name,ones,zeros,total,share_of_ones,note
0,Recurrent_Trader,128,157,285,0.449123,"codes {4,5}=1, {1,2,3}=0"
1,Day_Trader,170,115,285,0.596491,Copied from Day_Trade
2,Lottery_Stock_Trader,177,108,285,0.621053,Copied from Lottery_Stocks
3,Smoker,40,245,285,0.140351,Copied from Smoke
4,Alcohol_User,120,165,285,0.421053,Copied from Drink_Alcohol
5,Male_Investor,168,117,285,0.589474,Copied from Male_Dummy
6,Single_Investor,176,109,285,0.617544,Copied from Single_Dummy
7,Needs_Returns_For_Expenses,142,143,285,0.498246,Copied from Return_needed_for_exp
8,Derivative_User,150,135,285,0.526316,Copied from Derivative_Warrant
9,Margin_User,23,262,285,0.080702,Copied from Margin_Account


In [51]:
binary_csv = Path('binarized_data.csv')
summary_csv = Path('binarization_summary.csv')
binary.to_csv(binary_csv, index=False)
summary.to_csv(summary_csv, index=False)
binary_csv, summary_csv


(PosixPath('binarized_data.csv'), PosixPath('binarization_summary.csv'))